****

**Nombre:** Juan Martín Cusme Carrera

**Materia:** Recuperación de información

**Curso:** GR1CC

**Fecha:** 13/11/2026

****

# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

Se usan los 18 textos en inglés del Proyecto Gutenberg distribuidos por NLTK,
igual que en la práctica 03. El texto original se conserva en la tabla.

Se aplica el mismo preprocesamiento de la práctica 03 para obtener los
tokens normalizados de los documentos y las consultas.

In [1]:
from pathlib import Path
from collections import Counter
import re
import unicodedata
import numpy as np
import pandas as pd

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

# Si hacen falta en otro equipo: %pip install nltk numpy pandas scipy scikit-learn
import nltk
from nltk.corpus import gutenberg

nltk.data.path.insert(0, str(raiz / ".venv" / "nltk_data"))

def normalizar(texto):
    texto = unicodedata.normalize("NFC", texto).lower()
    return re.findall(r"[^\W\d_]+", texto)

filas_corpus = []
for id_doc in gutenberg.fileids():
    texto = gutenberg.raw(id_doc)
    encabezado = texto.splitlines()[0].strip().strip("[]")
    titulo = encabezado.split(" by ")[0].rstrip(",")
    filas_corpus.append([str(id_doc), titulo, texto, normalizar(texto)])

tabla_corpus = pd.DataFrame(filas_corpus, columns=["doc_id", "Título", "Texto", "tokens"])
with pd.option_context("display.max_colwidth", 55):
    display(tabla_corpus)

print("Número de documentos:", len(tabla_corpus))
print("Corpus local:", gutenberg.root)

,doc_id,Título,Texto,tokens
0,austen-emma.txt,Emma,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAPTER I...,"[emma, by, jane, austen, volume, i, chapter, i, emm..."
1,austen-persuasion.txt,Persuasion,[Persuasion by Jane Austen 1818]\n\n\nChapter 1\n\n...,"[persuasion, by, jane, austen, chapter, sir, walter..."
2,austen-sense.txt,Sense and Sensibility,[Sense and Sensibility by Jane Austen 1811]\n\nCHAP...,"[sense, and, sensibility, by, jane, austen, chapter..."
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of the ...,"[the, king, james, bible, the, old, testament, of, ..."
4,blake-poems.txt,Poems,[Poems by William Blake 1789]\n\n \nSONGS OF INNOCE...,"[poems, by, william, blake, songs, of, innocence, a..."
5,bryant-stories.txt,Stories to Tell to Children,[Stories to Tell to Children by Sara Cone Bryant 19...,"[stories, to, tell, to, children, by, sara, cone, b..."
6,burgess-busterbrown.txt,The Adventures of Buster Bear,[The Adventures of Buster Bear by Thornton W. Burge...,"[the, adventures, of, buster, bear, by, thornton, w..."
7,carroll-alice.txt,Alice's Adventures in Wonderland,[Alice's Adventures in Wonderland by Lewis Carroll ...,"[alice, s, adventures, in, wonderland, by, lewis, c..."
8,chesterton-ball.txt,The Ball and The Cross,[The Ball and The Cross by G.K. Chesterton 1909]\n\...,"[the, ball, and, the, cross, by, g, k, chesterton, ..."
9,chesterton-brown.txt,The Wisdom of Father Brown,[The Wisdom of Father Brown by G. K. Chesterton 191...,"[the, wisdom, of, father, brown, by, g, k, chestert..."


Número de documentos: 18
Corpus local: C:\Users\DETPC\Desktop\ir26b\.venv\nltk_data\corpora\gutenberg


DF cuenta los documentos en los que aparece cada término. Se usa
$IDF_{\mathrm{TFIDF}}(t)=1+\ln(N/df(t))$, igual que en la práctica 03. Para BM25
se utiliza la variante de IDF positiva $\ln(1+(N-df(t)+0.5)/(df(t)+0.5))$.
La longitud de un documento es su número de tokens, incluidas las
repeticiones. Todas estas estadísticas se calculan una sola vez.
El código calcula el IDF de BM25 a partir del DF, el número de documentos sin el término y el logaritmo natural.

In [2]:
from math import log1p

ids_documentos = tabla_corpus["doc_id"].tolist()
titulos = dict(zip(ids_documentos, tabla_corpus["Título"]))
tf_documentos = {}
longitudes = {}
df = Counter()

for id_doc, tokens in zip(ids_documentos, tabla_corpus["tokens"]):
    frecuencias = Counter(tokens)
    tf_documentos[id_doc] = frecuencias
    longitudes[id_doc] = len(tokens)
    df.update(frecuencias.keys())

N = len(ids_documentos)
longitud_media = sum(longitudes.values()) / N
vocabulario = sorted(df)
idf_tfidf = {}
idf_bm25 = {}

for termino in vocabulario:
    documentos_con_termino = df[termino]
    documentos_sin_termino = N - documentos_con_termino

    idf_tfidf[termino] = 1 + np.log(N / documentos_con_termino)
    cociente = (documentos_sin_termino + 0.5) / (documentos_con_termino + 0.5)
    idf_bm25[termino] = log1p(cociente)  # Logaritmo natural de 1 + cociente.

print("Tamaño del vocabulario:", len(vocabulario))
print(f"Longitud media del corpus: {longitud_media:.2f} tokens")

Tamaño del vocabulario: 41509
Longitud media del corpus: 118670.78 tokens


**2. Benchmarking**

Se seleccionaron ocho consultas en el idioma del corpus y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

Las primeras cuatro consultas se retoman de la práctica 03. Las otras
permiten probar un término presente en todo el corpus, uno presente en un
solo texto y búsquedas más largas. La tabla muestra el DF de cada palabra y
la necesidad de información de cada consulta.

In [3]:
consultas = [
    ("C1", "whale ship captain", "Encontrar relatos de ballenas, barcos y capitanes."),
    ("C2", "alice rabbit queen", "Encontrar la aventura de Alicia con el conejo y la reina."),
    ("C3", "love marriage family", "Buscar historias sobre amor, matrimonio y familia."),
    ("C4", "hamlet ghost revenge", "Encontrar pasajes sobre Hamlet, el fantasma y la venganza."),
    ("C5", "the", "Localizar textos donde se usa el artículo inglés the."),
    ("C6", "primroses", "Encontrar menciones a las prímulas."),
    ("C7", "the love of a family", "Buscar pasajes que traten del afecto familiar."),
    ("C8", "sailor ocean voyage storm whale ship", "Buscar viajes marítimos con marineros, tormentas y ballenas.")
]

tokens_consultas = {}
filas_consultas = []
for clave, texto, necesidad in consultas:
    tokens = normalizar(texto)
    tokens_consultas[clave] = tokens
    frecuencias_df = ", ".join(f"{termino} = {df[termino]}" for termino in tokens)
    filas_consultas.append([clave, texto, necesidad, len(tokens), frecuencias_df])

tabla_consultas = pd.DataFrame(filas_consultas, columns=["Consulta", "Texto", "Qué busca el usuario", "Número de tokens", "DF por término"])
with pd.option_context("display.max_colwidth", None):
    display(tabla_consultas)

,Consulta,Texto,Qué busca el usuario,Número de tokens,DF por término
0,C1,whale ship captain,"Encontrar relatos de ballenas, barcos y capitanes.",3,"whale = 6, ship = 13, captain = 9"
1,C2,alice rabbit queen,Encontrar la aventura de Alicia con el conejo y la reina.,3,"alice = 3, rabbit = 6, queen = 15"
2,C3,love marriage family,"Buscar historias sobre amor, matrimonio y familia.",3,"love = 15, marriage = 12, family = 13"
3,C4,hamlet ghost revenge,"Encontrar pasajes sobre Hamlet, el fantasma y la venganza.",3,"hamlet = 6, ghost = 9, revenge = 8"
4,C5,the,Localizar textos donde se usa el artículo inglés the.,1,the = 18
5,C6,primroses,Encontrar menciones a las prímulas.,1,primroses = 1
6,C7,the love of a family,Buscar pasajes que traten del afecto familiar.,5,"the = 18, love = 15, of = 18, a = 18, family = 13"
7,C8,sailor ocean voyage storm whale ship,"Buscar viajes marítimos con marineros, tormentas y ballenas.",6,"sailor = 6, ocean = 9, voyage = 8, storm = 12, whale = 6, ship = 13"


La consulta C5 usa "the", presente en los 18 textos. C6 usa "primroses",
presente en uno solo. C7 mezcla palabras que aparecen en toda la colección
con términos temáticos como "love" y "family". C8 combina seis términos con
distinto DF, por ejemplo "whale" (6) y "ship" (13).

In [4]:
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def devolver_tokens(tokens):
    return tokens

vectorizador = TfidfVectorizer(
    analyzer=devolver_tokens,
    vocabulary=vocabulario,
    lowercase=False,
    token_pattern=None,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)
matriz_tfidf = vectorizador.fit_transform(tabla_corpus["tokens"])

assert issparse(matriz_tfidf)
assert list(vectorizador.get_feature_names_out()) == vocabulario
assert np.allclose(vectorizador.idf_, [idf_tfidf[t] for t in vocabulario])
print("Matriz TF-IDF:", matriz_tfidf.shape, "| Formato:", matriz_tfidf.format)

def puntuar_tfidf(tokens_consulta):
    vector_consulta = vectorizador.transform([tokens_consulta])
    return cosine_similarity(vector_consulta, matriz_tfidf)[0]

Matriz TF-IDF: (18, 41509) | Formato: csr


El vectorizador se ajusta con los documentos. Cada consulta usa
"transform" y el mismo vocabulario e IDF. La matriz documento-término
se mantiene dispersa. El coseno devuelve una puntuación por documento.

En BM25 se suma un aporte por cada término distinto de la consulta que está
en el vocabulario. Para un término $t$ en un documento $d$ se calcula

$$
IDF_{\mathrm{BM25}}(t)
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1\left(1-b+b|d|/\overline{|d|}\right)}.
$$

$k_1$ controla la saturación de las repeticiones. $b$ controla cuánto
influye la longitud del documento respecto a la media. Se usa la forma
con el factor $(k_1+1)$ y el IDF positivo indicado en el apartado 1.

In [5]:
def aportes_bm25(tokens_consulta, id_doc, k1=1.2, b=0.75):
    factor_longitud = 1 - b + b * longitudes[id_doc] / longitud_media
    aportes = {}
    for termino in dict.fromkeys(tokens_consulta):
        if termino in df:
            frecuencia = tf_documentos[id_doc][termino]
            factor_tf = frecuencia * (k1 + 1) / (frecuencia + k1 * factor_longitud)
            aportes[termino] = idf_bm25[termino] * factor_tf
    return aportes

def puntuar_bm25(tokens_consulta, k1=1.2, b=0.75):
    puntuaciones = []
    for id_doc in ids_documentos:
        aportes = aportes_bm25(tokens_consulta, id_doc, k1, b)
        puntuaciones.append(sum(aportes.values()))
    return puntuaciones

def crear_ranking(puntuaciones, limite=10):
    tabla = pd.DataFrame({
        "doc_id": ids_documentos,
        "Título": [titulos[id_doc] for id_doc in ids_documentos],
        "Puntuación": puntuaciones
    })
    tabla = tabla[tabla["Puntuación"] > 0]
    tabla = tabla.sort_values(["Puntuación", "doc_id"], ascending=[False, True])
    tabla = tabla.head(limite).reset_index(drop=True)
    tabla.insert(0, "Posición", range(1, len(tabla) + 1))
    return tabla

In [6]:
rankings_tfidf = {}
rankings_bm25 = {}

for clave, tokens in tokens_consultas.items():
    rankings_tfidf[clave] = crear_ranking(puntuar_tfidf(tokens))
    rankings_bm25[clave] = crear_ranking(puntuar_bm25(tokens))

# Comprobar aparte el caso de una consulta sin términos del vocabulario.
tokens_ausentes = normalizar("qzxqzxqzxausente")
assert all(termino not in df for termino in tokens_ausentes)
sin_coincidencias_tfidf = crear_ranking(puntuar_tfidf(tokens_ausentes))
sin_coincidencias_bm25 = crear_ranking(puntuar_bm25(tokens_ausentes))
assert sin_coincidencias_tfidf.empty and sin_coincidencias_bm25.empty
print("Consulta sin términos del vocabulario:", len(sin_coincidencias_tfidf),
      "resultados TF-IDF y", len(sin_coincidencias_bm25), "resultados BM25.")

# Comprobar el efecto de descartar los dígitos en la consulta.
tokens_anio = normalizar("1816")
assert tokens_anio == []
ranking_anio_tfidf = crear_ranking(puntuar_tfidf(tokens_anio))
ranking_anio_bm25 = crear_ranking(puntuar_bm25(tokens_anio))
assert ranking_anio_tfidf.empty and ranking_anio_bm25.empty
print("Consulta por 1816:", len(tokens_anio), "tokens y",
      len(ranking_anio_tfidf), "resultados TF-IDF y",
      len(ranking_anio_bm25), "resultados BM25.")


Consulta sin términos del vocabulario: 0 resultados TF-IDF y 0 resultados BM25.
Consulta por 1816: 0 tokens y 0 resultados TF-IDF y 0 resultados BM25.


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [7]:
filas_resumen = []

for clave, texto, necesidad in consultas:
    ranking_tfidf = rankings_tfidf[clave]
    ranking_bm25 = rankings_bm25[clave]
    ids_tfidf = ranking_tfidf["doc_id"].tolist()
    ids_bm25 = ranking_bm25["doc_id"].tolist()

    comparacion = pd.DataFrame({"Posición": range(1, max(len(ids_tfidf), len(ids_bm25)) + 1)})
    comparacion["ID y título TF-IDF"] = ranking_tfidf["doc_id"] + " | " + ranking_tfidf["Título"]
    comparacion["Coseno"] = ranking_tfidf["Puntuación"]
    comparacion["ID y título BM25"] = ranking_bm25["doc_id"] + " | " + ranking_bm25["Título"]
    comparacion["Score BM25"] = ranking_bm25["Puntuación"]
    print(clave, "-", texto)
    with pd.option_context("display.max_colwidth", None, "display.precision", 9):
        display(comparacion)

    compartidos = len(set(ids_tfidf) & set(ids_bm25))
    primero_tfidf = ids_tfidf[0] if ids_tfidf else "Sin resultados"
    primero_bm25 = ids_bm25[0] if ids_bm25 else "Sin resultados"
    filas_resumen.append([clave, len(ids_tfidf), len(ids_bm25), compartidos,
                         primero_tfidf, primero_bm25, ids_tfidf == ids_bm25])

tabla_resumen = pd.DataFrame(filas_resumen, columns=[
    "Consulta", "Tamaño TF-IDF", "Tamaño BM25", "IDs en común",
    "Primer doc_id TF-IDF", "Primer doc_id BM25", "Mismo orden"
])
display(tabla_resumen)

C1 - whale ship captain


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt | Moby Dick,0.112064711,melville-moby_dick.txt | Moby Dick,4.620971754
1,2,austen-persuasion.txt | Persuasion,0.036287119,bryant-stories.txt | Stories to Tell to Children,4.081061900
2,3,bryant-stories.txt | Stories to Tell to Children,0.005457198,whitman-leaves.txt | Leaves of Grass,4.055815415
3,4,whitman-leaves.txt | Leaves of Grass,0.005125376,bible-kjv.txt | The King James Bible,2.910066042
4,5,chesterton-brown.txt | The Wisdom of Father Brown,0.003887573,shakespeare-hamlet.txt | The Tragedie of Hamlet,2.460251604
5,6,chesterton-ball.txt | The Ball and The Cross,0.002942016,chesterton-ball.txt | The Ball and The Cross,2.345901081
6,7,edgeworth-parents.txt | The Parent's Assistant,0.002743844,chesterton-brown.txt | The Wisdom of Father Brown,2.176922885
7,8,bible-kjv.txt | The King James Bible,0.001728034,austen-persuasion.txt | Persuasion,2.170419133
8,9,shakespeare-hamlet.txt | The Tragedie of Hamlet,0.001486211,edgeworth-parents.txt | The Parent's Assistant,1.961218490
9,10,chesterton-thursday.txt | The Man Who Was Thursday,0.000497700,chesterton-thursday.txt | The Man Who Was Thursday,1.658630978


C2 - alice rabbit queen


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,carroll-alice.txt | Alice's Adventures in Wonderland,0.310256747,carroll-alice.txt | Alice's Adventures in Wonderland,6.498416395
1,2,burgess-busterbrown.txt | The Adventures of Buster Bear,0.007132500,chesterton-thursday.txt | The Man Who Was Thursday,3.810676763
2,3,bryant-stories.txt | Stories to Tell to Children,0.002743386,bryant-stories.txt | Stories to Tell to Children,2.618796590
3,4,blake-poems.txt | Poems,0.002641260,edgeworth-parents.txt | The Parent's Assistant,2.485170181
4,5,shakespeare-hamlet.txt | The Tragedie of Hamlet,0.001099129,chesterton-ball.txt | The Ball and The Cross,2.279065489
5,6,edgeworth-parents.txt | The Parent's Assistant,0.000757436,burgess-busterbrown.txt | The Adventures of Buster Bear,2.272213086
6,7,chesterton-ball.txt | The Ball and The Cross,0.000736293,chesterton-brown.txt | The Wisdom of Father Brown,2.184540862
7,8,chesterton-thursday.txt | The Man Who Was Thursday,0.000657602,shakespeare-hamlet.txt | The Tragedie of Hamlet,0.420109718
8,9,chesterton-brown.txt | The Wisdom of Father Brown,0.000640492,blake-poems.txt | Poems,0.418412311
9,10,melville-moby_dick.txt | Moby Dick,0.000316809,melville-moby_dick.txt | Moby Dick,0.404010500


C3 - love marriage family


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,blake-poems.txt | Poems,0.026154704,austen-sense.txt | Sense and Sensibility,2.078348384
1,2,austen-persuasion.txt | Persuasion,0.013829925,austen-persuasion.txt | Persuasion,2.071503769
2,3,austen-sense.txt | Sense and Sensibility,0.013032315,austen-emma.txt | Emma,2.061976351
3,4,whitman-leaves.txt | Leaves of Grass,0.012918060,bible-kjv.txt | The King James Bible,1.845839292
4,5,austen-emma.txt | Emma,0.010585356,whitman-leaves.txt | Leaves of Grass,1.842627294
5,6,milton-paradise.txt | Paradise Lost,0.009993979,chesterton-thursday.txt | The Man Who Was Thursday,1.840524729
6,7,chesterton-brown.txt | The Wisdom of Father Brown,0.005699778,bryant-stories.txt | Stories to Tell to Children,1.670931161
7,8,edgeworth-parents.txt | The Parent's Assistant,0.005225586,milton-paradise.txt | Paradise Lost,1.544265167
8,9,bible-kjv.txt | The King James Bible,0.002941641,melville-moby_dick.txt | Moby Dick,1.526302891
9,10,bryant-stories.txt | Stories to Tell to Children,0.002763832,edgeworth-parents.txt | The Parent's Assistant,1.161774755


... (7 outputs left out)

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,chesterton-brown.txt | The Wisdom of Father Brown,0.491678788,austen-sense.txt | Sense and Sensibility,1.357890641
1,2,melville-moby_dick.txt | Moby Dick,0.483384519,austen-persuasion.txt | Persuasion,1.357206145
2,3,chesterton-thursday.txt | The Man Who Was Thursday,0.464317357,austen-emma.txt | Emma,1.355300528
3,4,chesterton-ball.txt | The Ball and The Cross,0.464118451,edgeworth-parents.txt | The Parent's Assistant,1.337718447
4,5,whitman-leaves.txt | Leaves of Grass,0.463216911,bible-kjv.txt | The King James Bible,1.330151630
5,6,blake-poems.txt | Poems,0.423233181,chesterton-brown.txt | The Wisdom of Father Brown,1.323379355
6,7,bible-kjv.txt | The King James Bible,0.419495377,bryant-stories.txt | Stories to Tell to Children,1.289966889
7,8,bryant-stories.txt | Stories to Tell to Children,0.415024198,chesterton-ball.txt | The Ball and The Cross,1.242009957
8,9,austen-persuasion.txt | Persuasion,0.384573020,chesterton-thursday.txt | The Man Who Was Thursday,1.239998748
9,10,edgeworth-parents.txt | The Parent's Assistant,0.382944731,melville-moby_dick.txt | Moby Dick,1.236584463


C8 - sailor ocean voyage storm whale ship


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt | Moby Dick,0.078394078,melville-moby_dick.txt | Moby Dick,9.502911700
1,2,whitman-leaves.txt | Leaves of Grass,0.007039185,whitman-leaves.txt | Leaves of Grass,8.777038815
2,3,bryant-stories.txt | Stories to Tell to Children,0.004137146,chesterton-ball.txt | The Ball and The Cross,6.007384036
3,4,chesterton-ball.txt | The Ball and The Cross,0.002461417,chesterton-brown.txt | The Wisdom of Father Brown,5.195708464
4,5,milton-paradise.txt | Paradise Lost,0.002348348,shakespeare-hamlet.txt | The Tragedie of Hamlet,4.856400156
5,6,chesterton-brown.txt | The Wisdom of Father Brown,0.001986347,bryant-stories.txt | Stories to Tell to Children,4.799448006
6,7,shakespeare-hamlet.txt | The Tragedie of Hamlet,0.001823211,milton-paradise.txt | Paradise Lost,4.028243872
7,8,austen-persuasion.txt | Persuasion,0.001060476,austen-persuasion.txt | Persuasion,2.690966410
8,9,shakespeare-caesar.txt | The Tragedie of Julius Caesar,0.000676547,edgeworth-parents.txt | The Parent's Assistant,2.632818011
9,10,blake-poems.txt | Poems,0.000636722,bible-kjv.txt | The King James Bible,2.328668733


,Consulta,Tamaño TF-IDF,Tamaño BM25,IDs en común,Primer doc_id TF-IDF,Primer doc_id BM25,Mismo orden
0,C1,10,10,10,melville-moby_dick.txt,melville-moby_dick.txt,False
1,C2,10,10,10,carroll-alice.txt,carroll-alice.txt,False
2,C3,10,10,8,blake-poems.txt,austen-sense.txt,False
3,C4,10,10,9,shakespeare-hamlet.txt,shakespeare-hamlet.txt,False
4,C5,10,10,8,whitman-leaves.txt,bible-kjv.txt,False
5,C6,1,1,1,edgeworth-parents.txt,edgeworth-parents.txt,True
6,C7,10,10,8,chesterton-brown.txt,austen-sense.txt,False
7,C8,10,10,8,melville-moby_dick.txt,melville-moby_dick.txt,False


C1 comparte los diez documentos entre los dos top 10 y los ordena de forma
distinta. En C6, ambos métodos devuelven un solo documento porque *primroses*
tiene DF igual a 1. Las puntuaciones se interpretan dentro de cada método.

In [8]:
tokens_ejemplo = tokens_consultas["C1"]
docs_ejemplo = ["austen-persuasion.txt", "bryant-stories.txt"]
filas_aportes = []
filas_posiciones = []
ranking_coseno = rankings_tfidf["C1"].set_index("doc_id")
ranking_base = rankings_bm25["C1"].set_index("doc_id")

for id_doc in docs_ejemplo:
    aportes = aportes_bm25(tokens_ejemplo, id_doc)
    for termino, aporte in aportes.items():
        filas_aportes.append([id_doc, longitudes[id_doc], termino,
                              tf_documentos[id_doc][termino], df[termino],
                              idf_bm25[termino], aporte])

    filas_posiciones.append([id_doc, ranking_coseno.loc[id_doc, "Posición"],
                             ranking_coseno.loc[id_doc, "Puntuación"],
                             ranking_base.loc[id_doc, "Posición"],
                             ranking_base.loc[id_doc, "Puntuación"]])
    assert np.isclose(sum(aportes.values()), ranking_base.loc[id_doc, "Puntuación"])

tabla_aportes = pd.DataFrame(filas_aportes, columns=[
    "doc_id", "Longitud |d|", "Término", "f(t,d)", "df(t)", "IDF BM25", "Aporte BM25"
])
tabla_posiciones = pd.DataFrame(filas_posiciones, columns=[
    "doc_id", "Posición TF-IDF", "Coseno", "Posición BM25", "Score BM25"
])
display(tabla_aportes.round(6))
display(tabla_posiciones.round(6))

,doc_id,Longitud |d|,Término,"f(t,d)",df(t),IDF BM25,Aporte BM25
0,austen-persuasion.txt,84123,whale,0,6,1.072637,0.000000
1,austen-persuasion.txt,84123,ship,6,13,0.341749,0.650201
2,austen-persuasion.txt,84123,captain,303,9,0.693147,1.520218
3,bryant-stories.txt,46697,whale,16,6,1.072637,2.267111
4,bryant-stories.txt,46697,ship,5,13,0.341749,0.664864
5,bryant-stories.txt,46697,captain,2,9,0.693147,1.149086


,doc_id,Posición TF-IDF,Coseno,Posición BM25,Score BM25
0,austen-persuasion.txt,2,0.036287,8,2.170419
1,bryant-stories.txt,3,0.005457,2,4.081062


Para C1, el coseno coloca a *Persuasion* en el puesto 2 (0,036287) y a *Stories to Tell to Children* en el puesto 3 (0,005457). En *Persuasion*, *captain* aparece 303 veces y tiene un peso TF-IDF alto que contribuye al producto punto con la consulta.

BM25 invierte ese orden. *Stories to Tell to Children* queda en el puesto 2 (4,081062) y *Persuasion* en el 8 (2,170419). El primer texto contiene los tres términos y es más corto (46.697 tokens frente a 84.123). Sus 16 menciones de *whale* aportan 2,267111 al score. *Whale* tiene el IDF BM25 más alto de los tres términos porque aparece en seis documentos. En *Persuasion*, las 303 repeticiones de *captain* se acercan al límite de saturación y la ausencia de *whale* deja ese aporte en cero.


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

Se mantiene C1 para examinar los parámetros. Se muestran posiciones dentro
del ranking completo de documentos con puntuación positiva, así que se puede
seguir a los mismos documentos si alguno pasa del décimo puesto.

In [9]:
configuraciones = [
    ("Base", 1.2, 0.75),
    ("Sin ajuste por longitud", 1.2, 0.00),
    ("Mayor efecto de las repeticiones", 2.0, 0.75)
]
docs_parametros = [
    "melville-moby_dick.txt",
    "bryant-stories.txt",
    "whitman-leaves.txt",
    "bible-kjv.txt",
    "austen-persuasion.txt"
]
filas_parametros = []
filas_aportes_parametros = []

for nombre, k1, b in configuraciones:
    puntuaciones = puntuar_bm25(tokens_ejemplo, k1, b)
    ranking = crear_ranking(puntuaciones, limite=N).set_index("doc_id")
    for id_doc in docs_parametros:
        posicion = ranking.loc[id_doc, "Posición"]
        puntuacion = ranking.loc[id_doc, "Puntuación"]
        filas_parametros.append([id_doc, longitudes[id_doc], nombre, k1, b, posicion, puntuacion])
        aportes = aportes_bm25(tokens_ejemplo, id_doc, k1, b)
        for termino, aporte in aportes.items():
            filas_aportes_parametros.append([id_doc, termino, tf_documentos[id_doc][termino],
                                             nombre, aporte])
        assert np.isclose(sum(aportes.values()), puntuacion)

tabla_parametros = pd.DataFrame(filas_parametros, columns=[
    "doc_id", "Longitud |d|", "Ejecución", "k1", "b", "Posición", "Score BM25"
])
orden_configuraciones = [nombre for nombre, k1, b in configuraciones]
posiciones_parametros = tabla_parametros.pivot(index=["doc_id", "Longitud |d|"], columns="Ejecución", values="Posición")
display(posiciones_parametros[orden_configuraciones].reindex(docs_parametros, level="doc_id"))

scores_parametros = tabla_parametros.pivot(index="doc_id", columns="Ejecución", values="Score BM25")
display(scores_parametros.loc[docs_parametros, orden_configuraciones].round(6))

tabla_aportes_parametros = pd.DataFrame(filas_aportes_parametros, columns=[
    "doc_id", "Término", "f(t,d)", "Ejecución", "Aporte"
])
comparacion_aportes = tabla_aportes_parametros.pivot(index=["doc_id", "Término", "f(t,d)"], columns="Ejecución", values="Aporte")
display(comparacion_aportes[orden_configuraciones].reindex(docs_parametros, level="doc_id").round(6))

,Ejecución,Base,Sin ajuste por longitud,Mayor efecto de las repeticiones
doc_id,Longitud |d|,,,
melville-moby_dick.txt,218382,1,1,1
bryant-stories.txt,46697,2,4,2
whitman-leaves.txt,126285,3,2,3
bible-kjv.txt,791842,4,3,4
austen-persuasion.txt,84123,8,6,7


Ejecución,Base,Sin ajuste por longitud,Mayor efecto de las repeticiones
doc_id,,,
melville-moby_dick.txt,4.620972,4.626986,6.287248
bryant-stories.txt,4.081062,3.754570,5.200148
whitman-leaves.txt,4.055815,4.077998,5.138896
bible-kjv.txt,2.910066,3.936796,3.540963
austen-persuasion.txt,2.170419,2.145449,2.882100


Ejecución                                  Base  Sin ajuste por longitud  \
doc_id                 Término f(t,d)                                      
melville-moby_dick.txt captain 329     1.515910                 1.519382   
                       ship    518     0.749020                 0.750111   
                       whale   1226    2.356042                 2.357493   
bryant-stories.txt     captain 2       1.149086                 0.953077   
                       ship    5       0.664864                 0.606329   
                       whale   16      2.267111                 2.195164   
whitman-leaves.txt     captain 19      1.430246                 1.434334   
                       ship    79      0.740066                 0.740599   
                       whale   5       1.885504                 1.903065   
bible-kjv.txt          captain 139     1.458752                 1.511872   
                       ship    71      0.690525                 0.739352   
                       whale   3       0.760790                 1.685572   
austen-persuasion.txt  captain 303     1.520218                 1.518908   
                       ship    6       0.650201                 0.626540   
                       whale   0       0.000000                 0.000000   

Ejecución                              Mayor efecto de las repeticiones  
doc_id                 Término f(t,d)                                    
melville-moby_dick.txt captain 329                             2.059037  
                       ship    518                             1.018835  
                       whale   1226                            3.209376  
bryant-stories.txt     captain 2                               1.345808  
                       ship    5                               0.841712  
                       whale   16                              3.012628  
whitman-leaves.txt     captain 19                              1.872816  
                       ship    79                              0.998746  
                       whale   5                               2.267334  
bible-kjv.txt          captain 139                             1.933279  
                       ship    71                              0.893063  
                       whale   3                               0.714621  
austen-persuasion.txt  captain 303                             2.068768  
                       ship    6                               0.813332  
                       whale   0                               0.000000

Efecto de $b$. Al fijar $b=0$, BM25 deja de ajustar la puntuación según la longitud de los textos. Stories to Tell to Children, que es más corto que la media, baja del segundo al cuarto puesto. The King James Bible, que es más largo, sube un puesto.

Efecto de $k_1$. Con $b$ fijo, aumentar $k_1$ hace que el efecto de las repeticiones tarde más en saturarse. Persuasion sube un puesto y Moby Dick sigue primero. En The King James Bible, el aporte de captain aumenta y el de whale disminuye. Su puntuación total sube, pero sigue en el cuarto puesto porque queda entre las puntuaciones de los mismos libros que la rodeaban.
